In [1]:
import argparse
import copy
import os
import os.path as osp
import time
import warnings
from logging import log

import mmcv
import torch
from mmcv import Config, DictAction
from mmcv.runner import get_dist_info, init_dist
from mmcv.utils import get_git_hash
from mmdet import __version__
from mmdet.models import build_detector
from mmdet.utils import collect_env

from ssod.apis import get_root_logger, set_random_seed, train_detector
from ssod.datasets import build_dataset
from ssod.utils import patch_config

In [2]:
config = "/rsrch5/home/trans_mol_path/cercan/data/acformer/checkpoints/lizard/ACFormer_Lizard_finetune_batch2.py"
cfg = Config.fromfile(config)


In [4]:
model = build_detector(
        cfg.model, train_cfg=cfg.get("train_cfg"), test_cfg=cfg.get("test_cfg")
    )
model.init_weights()

/App/ACFormer/ssod/models/global_local_stn_sequence_plus.py:71: UserWarning: nn.init.uniform is now deprecated in favor of nn.init.uniform_.
  torch.nn.init.uniform(m.weight, a=-0.1, b=0.1)
2024-09-20 21:43:39,377 - mmcv - INFO - initialize ConvNeXt with init_cfg {'type': 'Pretrained', 'checkpoint': '/rsrch5/home/trans_mol_path/cercan/data/acformer/checkpoints/lizard/ACFormer_Lizard_BL_AAT_GL.pth', 'prefix': None}
2024-09-20 21:43:39,379 - mmcv - INFO - load model from: /rsrch5/home/trans_mol_path/cercan/data/acformer/checkpoints/lizard/ACFormer_Lizard_BL_AAT_GL.pth
2024-09-20 21:43:39,380 - mmcv - INFO - load checkpoint from local path: /rsrch5/home/trans_mol_path/cercan/data/acformer/checkpoints/lizard/ACFormer_Lizard_BL_AAT_GL.pth
2024-09-20 21:43:52,738 - mmcv - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: absolute_pos_embed, affine_token, globals.backbone.downsample_layers.0.0.weight, globals.backbone.downsample_layers.0.0.bi

In [6]:
from mmcv.runner import load_checkpoint
load_checkpoint(model, '/rsrch5/home/trans_mol_path/cercan/data/acformer/checkpoints/lizard/ACFormer_Lizard_BL_AAT_GL.pth')

load checkpoint from local path: /rsrch5/home/trans_mol_path/cercan/data/acformer/checkpoints/lizard/ACFormer_Lizard_BL_AAT_GL.pth
The model and loaded state dict do not match exactly

missing keys in source state_dict: norm.weight, norm.bias



{'meta': {'env_info': 'sys.platform: linux\nPython: 3.8.13 (default, Mar 28 2022, 11:38:47) [GCC 7.5.0]\nCUDA available: True\nGPU 0: Tesla V100S-PCIE-32GB\nCUDA_HOME: /cm/shared/apps/cuda10.2/toolkit/10.2.89\nNVCC: Cuda compilation tools, release 10.2, V10.2.8\nGCC: gcc (GCC) 11.2.0\nPyTorch: 1.12.1+cu102\nPyTorch compiling details: PyTorch built with:\n  - GCC 7.3\n  - C++ Version: 201402\n  - Intel(R) Math Kernel Library Version 2020.0.0 Product Build 20191122 for Intel(R) 64 architecture applications\n  - Intel(R) MKL-DNN v2.6.0 (Git Hash 52b5f107dd9cf10910aaa19cb47f3abf9b349815)\n  - OpenMP 201511 (a.k.a. OpenMP 4.5)\n  - LAPACK is enabled (usually provided by MKL)\n  - NNPACK is enabled\n  - CPU capability usage: AVX2\n  - CUDA Runtime 10.2\n  - NVCC architecture flags: -gencode;arch=compute_37,code=sm_37;-gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_70,code=sm_70\n  - CuDNN 7.6.5\n  - Magma 2.5.2\n  - Build settings: BLAS_INFO=mkl,

In [10]:
for name, param in model.named_parameters():
    if not param.requires_grad:
        print(f"Layer {name} is not trainable.")


Layer globals.backbone.downsample_layers.0.0.weight is not trainable.
Layer globals.backbone.downsample_layers.0.0.bias is not trainable.
Layer globals.backbone.downsample_layers.0.1.weight is not trainable.
Layer globals.backbone.downsample_layers.0.1.bias is not trainable.
Layer globals.backbone.downsample_layers.1.0.weight is not trainable.
Layer globals.backbone.downsample_layers.1.0.bias is not trainable.
Layer globals.backbone.downsample_layers.1.1.weight is not trainable.
Layer globals.backbone.downsample_layers.1.1.bias is not trainable.
Layer globals.backbone.downsample_layers.2.0.weight is not trainable.
Layer globals.backbone.downsample_layers.2.0.bias is not trainable.
Layer globals.backbone.downsample_layers.2.1.weight is not trainable.
Layer globals.backbone.downsample_layers.2.1.bias is not trainable.
Layer globals.backbone.downsample_layers.3.0.weight is not trainable.
Layer globals.backbone.downsample_layers.3.0.bias is not trainable.
Layer globals.backbone.downsample_